<a href="https://colab.research.google.com/github/lydiako861-a11y/Rust-conda/blob/main/Condor_Hummingbot_Feeder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
import time
import hashlib
from decimal import Decimal
from typing import Dict, Any

# Hummingbot imports
from hummingbot.strategy.script_strategy_base import ScriptStrategyBase
from hummingbot.core.data_type.common import OrderType, PositionAction, PositionSide
from hummingbot.core.utils.tracking_nonce import get_tracking_nonce

# Third-party imports (ensure redis is installed in your hummingbot environment: pip install redis)
try:
    import redis
except ImportError:
    raise ImportError("Please install redis: pip install redis")

class CondorOpportunityFeeder(ScriptStrategyBase):
    """
    Condor Opportunity Feeder
    -------------------------
    This script runs inside Hummingbot. It monitors a CEX (e.g., Binance)
    and a DEX (e.g., Uniswap) for price spreads. When a spread exceeds the
    MIN_PROFITABILITY_BPS threshold, it packages the data and pushes it to
    the Redis Opportunity Queue for the Condor Multi-Agent Cluster to execute
    via the CondorArbitrage.sol flash loan contract.
    """

    # --- Configuration ---
    # Define the markets to monitor
    CEX_EXCHANGE = "binance"
    DEX_EXCHANGE = "uniswap_v3"
    TRADING_PAIR = "WETH-USDC"

    markets = {
        CEX_EXCHANGE: {TRADING_PAIR},
        DEX_EXCHANGE: {TRADING_PAIR}
    }

    # Minimum spread in basis points (1 bps = 0.01%) required to trigger an alert
    MIN_PROFITABILITY_BPS = Decimal("15.0")

    # Trade size to simulate for the opportunity
    TRADE_SIZE_WETH = Decimal("10.0")

    def __init__(self, connectors: Dict[str, Any]):
        super().__init__(connectors)

        # Initialize Redis connection (Fallback to localhost if env var is missing)
        redis_url = os.getenv("REDIS_URL", "redis://localhost:6379/0")
        self.logger().info(f"Connecting to Redis Opportunity Queue at {redis_url}...")
        self.redis_client = redis.from_url(redis_url, decode_responses=True)

        self.last_opportunity_time = 0
        self.cooldown_seconds = 5  # Prevent spamming the same opportunity

    def on_tick(self):
        """
        This runs every tick (usually every second).
        We calculate the price difference between the CEX and DEX.
        """
        # Ensure connectors are ready
        if not self.connectors_ready:
            return

        # Fetch best ask and best bid from both exchanges
        cex_ask = self.connectors[self.CEX_EXCHANGE].get_price(self.TRADING_PAIR, True)
        cex_bid = self.connectors[self.CEX_EXCHANGE].get_price(self.TRADING_PAIR, False)

        dex_ask = self.connectors[self.DEX_EXCHANGE].get_price(self.TRADING_PAIR, True)
        dex_bid = self.connectors[self.DEX_EXCHANGE].get_price(self.TRADING_PAIR, False)

        # Scenario 1: Buy on CEX, Sell on DEX
        spread_cex_dex = ((dex_bid - cex_ask) / cex_ask) * Decimal("10000")

        # Scenario 2: Buy on DEX, Sell on CEX
        spread_dex_cex = ((cex_bid - dex_ask) / dex_ask) * Decimal("10000")

        current_time = time.time()

        # Evaluate Scenario 1
        if spread_cex_dex >= self.MIN_PROFITABILITY_BPS and (current_time - self.last_opportunity_time > self.cooldown_seconds):
            self.push_to_condor_queue(
                buy_exchange=self.CEX_EXCHANGE,
                sell_exchange=self.DEX_EXCHANGE,
                buy_price=cex_ask,
                sell_price=dex_bid,
                spread_bps=spread_cex_dex
            )
            self.last_opportunity_time = current_time

        # Evaluate Scenario 2
        elif spread_dex_cex >= self.MIN_PROFITABILITY_BPS and (current_time - self.last_opportunity_time > self.cooldown_seconds):
            self.push_to_condor_queue(
                buy_exchange=self.DEX_EXCHANGE,
                sell_exchange=self.CEX_EXCHANGE,
                buy_price=dex_ask,
                sell_price=cex_bid,
                spread_bps=spread_dex_cex
            )
            self.last_opportunity_time = current_time

    def push_to_condor_queue(self, buy_exchange: str, sell_exchange: str, buy_price: Decimal, sell_price: Decimal, spread_bps: Decimal):
        """
        Packages the arbitrage opportunity and pushes it to Redis.
        """
        opportunity = {
            "timestamp": time.time(),
            "pair": self.TRADING_PAIR,
            "buy_exchange": buy_exchange,
            "sell_exchange": sell_exchange,
            "buy_price": float(buy_price),
            "sell_price": float(sell_price),
            "amount": float(self.TRADE_SIZE_WETH),
            "expected_profit_bps": float(spread_bps),
            "target_contract": "CondorArbitrage" # Signals the agent to use the flash loan contract
        }

        # Generate a unique ID for this opportunity
        opp_id = hashlib.md5(json.dumps(opportunity, sort_keys=True).encode()).hexdigest()

        # Priority can be based on spread size (higher spread = higher priority)
        # We negate the spread because Redis ZADD sorts lowest to highest by default
        # (or we can just use time and let the agent sort)
        score = time.time() - (float(spread_bps) / 100)

        try:
            # Add to the sorted set
            self.redis_client.zadd("opportunities", {opp_id: score})
            # Store the actual payload
            self.redis_client.hset("opportunity_details", opp_id, json.dumps(opportunity))

            self.logger().info(f"🚀 Pushed Arbitrage Opportunity to Redis! Spread: {spread_bps:.2f} bps | Buy: {buy_exchange} | Sell: {sell_exchange}")
        except Exception as e:
            self.logger().error(f"Failed to push to Redis: {e}")

    def format_status(self) -> str:
        """Displays status in the Hummingbot CLI."""
        if not self.connectors_ready:
            return "Warming up connectors..."

        cex_ask = self.connectors[self.CEX_EXCHANGE].get_price(self.TRADING_PAIR, True)
        dex_bid = self.connectors[self.DEX_EXCHANGE].get_price(self.TRADING_PAIR, False)

        spread = ((dex_bid - cex_ask) / cex_ask) * Decimal("10000")

        return f"Condor Feeder Active | Monitoring {self.TRADING_PAIR} | Current Spread: {spread:.2f} bps | Target: {self.MIN_PROFITABILITY_BPS} bps"

ModuleNotFoundError: No module named 'hummingbot'